In [70]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen3:4b", temperature=0, reasoning=True)

reponse = llm.invoke("Dis bonjour en une phrase.")

print("RÉPONSE :", reponse.content)
print("RAISONNEMENT :", reponse.additional_kwargs.get("reasoning_content", "(aucun)")[:200])

RÉPONSE : Bonjour !
RAISONNEMENT : Okay, the user asked "Dis bonjour en une phrase." which translates to "Say hello in one sentence." in French.

First, I need to make sure I understand the request correctly. They want a greeting in a 


In [71]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="Tu es un assistant utile. Réponds en français, de façon courte."
)

In [72]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "Je m'appelle Houssam"}]
})
print(response["messages"][-1].content)

Bonjour Houssam ! 😊


In [73]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "Quel est mon prénom ?"}]
})
print(response["messages"][-1].content)

Je n'ai pas accès à votre prénom.


In [74]:
from langgraph.checkpoint.memory import InMemorySaver

agent_memoire = create_agent(
    model=llm,
    tools=[],
    system_prompt="Tu es un assistant utile. Réponds en français, de façon courte.",
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "1"}}

In [75]:
res = agent_memoire.invoke(
    {"messages": [{"role": "user", "content": "Je m'appelle Houssam"}]},
    config=config
)
print(res["messages"][-1].content)

Bonjour Houssam ! Enchanté.


In [76]:
res = agent_memoire.invoke(
    {"messages": [{"role": "user", "content": "Quel est mon prénom ?"}]},
    config=config
)
print(res["messages"][-1].content)

Houssam


In [77]:
config2 = {"configurable": {"thread_id": "2"}}

res = agent_memoire.invoke(
    {"messages": [{"role": "user", "content": "Quel est mon prénom ?"}]},
    config=config2
)
print(res["messages"][-1].content)

Je n'ai pas accès à votre prénom.


## Étape 2 : Tools personnalisés

In [78]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Donne la météo actuelle d'une ville."""
    print(f"[TOOL] get_weather appelé pour {location}")
    return f"Météo à {location} : ensoleillé, 32°C"

@tool
def get_employee_info(name: str) -> dict:
    """Donne le salaire et le poste d'un employé à partir de son prénom."""
    print(f"[TOOL] get_employee_info appelé pour {name}")
    return {"name": name, "salary": 12000, "job": "Développeur"}

@tool
def multiply(a: float, b: float) -> float:
    """Multiplie deux nombres."""
    print(f"[TOOL] multiply appelé avec a={a}, b={b}")
    return a * b

tools = [get_weather, get_employee_info, multiply]

In [79]:
print(get_weather.invoke({"location": "Marrakech"}))
print(multiply.invoke({"a": 6, "b": 7}))
print(get_weather.name, "-", get_weather.description)

[TOOL] get_weather appelé pour Marrakech
Météo à Marrakech : ensoleillé, 32°C
[TOOL] multiply appelé avec a=6.0, b=7.0
42.0
get_weather - Donne la météo actuelle d'une ville.


In [80]:
from langgraph.checkpoint.memory import InMemorySaver

agent_tools = create_agent(
    model=llm,
    tools=tools,
    system_prompt="Tu es un assistant utile. Utilise les outils quand c'est nécessaire. Réponds en français.",
    checkpointer=InMemorySaver()
)

config_tools = {"configurable": {"thread_id": "tools-1"}}

In [81]:
res = agent_tools.invoke(
    {"messages": [{"role": "user", "content": "Quelle est la météo à Casablanca ?"}]},
    config=config_tools
)
print(res["messages"][-1].content)

[TOOL] get_weather appelé pour Casablanca
La météo à Casablanca est actuellement ensoleillée à 32°C.


In [82]:
res = agent_tools.invoke(
    {"messages": [{"role": "user", "content": "Quel est le salaire de Mohamed ? Multiplie-le par 3."}]},
    config=config_tools
)
print(res["messages"][-1].content)

[TOOL] get_employee_info appelé pour Mohamed
[TOOL] multiply appelé avec a=12000.0, b=3.0
Le salaire de Mohamed est de 12 000. En multipliant par 3, on obtient **36 000**.


## Étape 3 : Tools prédéfinis

In [83]:
from ddgs import DDGS

for backend in ["duckduckgo", "brave", "bing", "google", "wikipedia"]:
    try:
        r = DDGS(timeout=20).text("capitale du Maroc", max_results=2, backend=backend)
        print(f"OK   {backend} : {len(r)} résultats - {r[0]['title'] if r else ''}")
    except Exception as e:
        print(f"FAIL {backend} : {str(e)[:80]}")

FAIL duckduckgo : No results found.
FAIL brave : No results found.
FAIL bing : No results found.
OK   google : 2 résultats - Rabat - Wikipedia
FAIL wikipedia : No results found.


In [84]:
from ddgs import DDGS
from langchain.tools import tool

@tool
def ddg_bing(query: str) -> str:
    """Recherche sur le web (via DuckDuckGo/Bing) et renvoie les premiers résultats."""
    r = DDGS(timeout=20).text(query, max_results=3, backend="bing")
    return "\n".join(f"{x['title']} - {x['href']}\n{x['body']}" for x in r)

print(ddg_bing.invoke({"query": "Quelle est la capitale du Maroc ?"})[:400])

Maroc — WikipédiaQuelle est la capitale du Maroc - moroccoanswers.comQuelle est la capitale du Maroc - memoriser-culture-generale.frCapitale du Maroc : Rabat, Ville Impériale et Moderne | 2026Capitale du Maroc : Rabat, ambassades, superficie et population - https://fr.wikipedia.org/wiki/Maroc
La capitale administrative et politique du Maroc est Rabat tandis que la capitale économique et la plus gr


In [85]:
from dotenv import load_dotenv
from langchain_tavily import TavilySearch

load_dotenv(override=True)

tavily = TavilySearch(max_results=3)
resultat = tavily.invoke({"query": "Quelle est la capitale du Maroc ?"})

for r in resultat["results"]:
    print("-", r["title"])
    print("  ", r["url"])

- Quelle est la capitale du Maroc ?
   https://darijaschool.com/quelle-est-la-capitale-du-maroc
- Rabat, capitale du Maroc : histoire et 6 lieux incontournables
   https://miglioritourmarocco.com/fr/capitale-du-Maroc
- Le Maroc en bref (superficie, grandes villes, etc.) | Guide de voyage - TraceDirecte
   https://www.tracedirecte.com/destinations/voyage-maroc/voyage/guides/le-maroc-en-bref


In [86]:
from langchain_experimental.tools import PythonREPLTool

python_repl = PythonREPLTool()
print(python_repl.invoke("print(2 ** 10)"))

1024



In [87]:
from langgraph.checkpoint.memory import InMemorySaver

tools_pre = [tavily, python_repl]

agent_pre = create_agent(
    model=llm,
    tools=tools_pre,
    system_prompt=(
        "Tu es un assistant utile. Utilise l'outil de recherche web pour les "
        "informations récentes, et l'outil Python pour les calculs. Réponds en français."
    ),
    checkpointer=InMemorySaver()
)

config_pre = {"configurable": {"thread_id": "pre-1"}}

In [88]:
def poser(question):
    res = agent_pre.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config_pre
    )
    for m in res["messages"]:
        for tc in getattr(m, "tool_calls", []) or []:
            print("[APPEL OUTIL]", tc["name"], tc["args"])
    print("\nRÉPONSE :", res["messages"][-1].content)

In [89]:
poser("Cherche sur le web quelle est la capitale du Maroc et donne-moi une source.")

[APPEL OUTIL] tavily_search {'query': 'capitale du Maroc'}

RÉPONSE : La capitale du Maroc est **Rabat**.  
Source : [Wikipédia (fr)](https://fr.wikipedia.org/wiki/Maroc) (article "Maroc").


In [90]:
poser("Calcule la factorielle de 10 en utilisant Python.")

[APPEL OUTIL] tavily_search {'query': 'capitale du Maroc'}
[APPEL OUTIL] Python_REPL {'query': 'import math; print(math.factorial(10))'}

RÉPONSE : La factorielle de 10 est **3628800**.  
Calcul effectué avec Python via la fonction `math.factorial(10)`.


In [91]:
poser("Calcule la factorielle de 10 en utilisant Python.")

[APPEL OUTIL] tavily_search {'query': 'capitale du Maroc'}
[APPEL OUTIL] Python_REPL {'query': 'import math; print(math.factorial(10))'}

RÉPONSE : La factorielle de 10 est **3628800**.  
Calcul effectué avec Python via la fonction `math.factorial(10)`.


## Étape 4 : Middlewares

### Middleware 1 : dynamic_model
Choisit le petit ou le grand modèle selon la longueur de la conversation.

In [92]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

petit_modele = ChatOllama(model="qwen3:1.7b", temperature=0, reasoning=True)
grand_modele = ChatOllama(model="qwen3:4b", temperature=0, reasoning=True)

@wrap_model_call
def choix_modele(request: ModelRequest, handler) -> ModelResponse:
    """Petit modèle au début de la conversation, grand modèle ensuite."""
    nb = len(request.state["messages"])
    if nb > 4:
        modele, nom = grand_modele, "qwen3:4b"
    else:
        modele, nom = petit_modele, "qwen3:1.7b"
    print(f"[MIDDLEWARE] {nb} messages -> {nom}")
    return handler(request.override(model=modele))

In [93]:
agent_dyn = create_agent(
    model=grand_modele,
    tools=[get_weather, multiply],
    system_prompt="Tu es un assistant utile. Réponds en français, de façon courte.",
    middleware=[choix_modele],
    checkpointer=InMemorySaver()
)

config_dyn = {"configurable": {"thread_id": "dyn-1"}}

In [94]:
for q in ["Bonjour !", "Quelle est la météo à Rabat ?", "Multiplie 8 par 7."]:
    res = agent_dyn.invoke({"messages": [{"role": "user", "content": q}]}, config=config_dyn)
    print("Q :", q)
    print("R :", res["messages"][-1].content, "\n")

[MIDDLEWARE] 1 messages -> qwen3:1.7b
Q : Bonjour !
R : Bonjour ! Je suis ici pour vous aider. Comment puis-je vous aider aujourd'hui ? 

[MIDDLEWARE] 3 messages -> qwen3:1.7b
[TOOL] get_weather appelé pour Rabat
[MIDDLEWARE] 5 messages -> qwen3:4b
Q : Quelle est la météo à Rabat ?
R : La météo à Rabat est ensoleillée à 32°C. 

[MIDDLEWARE] 7 messages -> qwen3:4b
[TOOL] multiply appelé avec a=8.0, b=7.0
[MIDDLEWARE] 9 messages -> qwen3:4b
Q : Multiplie 8 par 7.
R : 8 fois 7 égale 56. 



### Middleware 2 : dynamic_prompt
Adapte le prompt système selon le rôle de l'utilisateur (expert ou débutant).

In [95]:
from typing import TypedDict
from langchain.agents.middleware import dynamic_prompt, ModelRequest

class Context(TypedDict):
    user_role: str

@dynamic_prompt
def prompt_selon_role(request: ModelRequest) -> str:
    """Génère le prompt système selon le rôle de l'utilisateur."""
    role = request.runtime.context.get("user_role", "user")
    base = "Tu es un assistant utile. Réponds en français."
    if role == "expert":
        prompt = f"{base} Donne une réponse technique et détaillée, en 4 phrases maximum."
    elif role == "beginner":
        prompt = f"{base} Explique simplement, sans jargon, avec une analogie, en 3 phrases maximum."
    else:
        prompt = base
    print(f"[PROMPT] rôle = {role}")
    return prompt

In [96]:
agent_prompt = create_agent(
    model=llm,
    tools=[],
    middleware=[prompt_selon_role],
    context_schema=Context,
)

In [97]:
question = "Explique ce qu'est le machine learning."

for role in ["expert", "beginner"]:
    res = agent_prompt.invoke(
        {"messages": [{"role": "user", "content": question}]},
        context={"user_role": role}
    )
    print(f"\n===== {role.upper()} =====")
    print(res["messages"][-1].content)

[PROMPT] rôle = expert

===== EXPERT =====
L'apprentissage automatique (ML) est une branche de l'intelligence artificielle permettant aux systèmes d'apprendre à partir de données pour identifier des motifs et effectuer des prédictions. Ces modèles utilisent des algorithmes statistiques pour analyser des ensembles de données complexes et extraire des patterns significatifs sans instructions explicites. L'objectif est de former un modèle capable d'améliorer sa précision et sa performance via des itérations d'entraînement, en s'adaptant aux nouvelles informations. Contrairement à la programmation classique, l'apprentissage automatique optimise automatiquement ses résultats en évitant la réécriture manuelle du code.
[PROMPT] rôle = beginner

===== BEGINNER =====
L'apprentissage automatique est une méthode où les ordinateurs apprennent à partir d'expériences sans être programmés à l'avance. C'est comme un enfant qui apprend à reconnaître des objets en regardant des exemples, sans avoir à êt

### Middleware 3 : gestion des erreurs d'outils

In [98]:
from langchain.tools import tool
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

@tool
def diviser(a: float, b: float) -> float:
    """Divise a par b."""
    return a / b

@wrap_tool_call
def gerer_erreurs_outils(request, handler):
    """Intercepte les erreurs d'outils et renvoie un message clair au modèle."""
    try:
        return handler(request)
    except Exception as e:
        print(f"[ERREUR OUTIL] {type(e).__name__} : {e}")
        return ToolMessage(
            content=f"Erreur de l'outil : {e}. Explique le problème à l'utilisateur.",
            tool_call_id=request.tool_call["id"]
        )

In [99]:
agent_err = create_agent(
    model=llm,
    tools=[diviser],
    system_prompt="Tu es un assistant utile. Utilise l'outil pour les divisions. Réponds en français, de façon courte.",
    middleware=[gerer_erreurs_outils],
)

In [100]:
res = agent_err.invoke(
    {"messages": [{"role": "user", "content": "Divise 10 par 0."}]}
)
print(res["messages"][-1].content)

[ERREUR OUTIL] ZeroDivisionError : float division by zero
Impossible de diviser par zéro.


### Middleware 4 : garde-fous (guardrails)

In [101]:
from typing import Any
from langchain.agents.middleware import before_agent, AgentState
from langgraph.runtime import Runtime

mots_interdits = ["hack", "exploit", "malware"]

@before_agent(can_jump_to=["end"])
def filtre_contenu(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Bloque les demandes contenant un mot interdit, avant tout traitement."""
    if not state["messages"]:
        return None

    premier = state["messages"][0]
    if premier.type != "human":
        return None

    texte = premier.content.lower()
    for mot in mots_interdits:
        if mot in texte:
            print(f"[GARDE-FOU] demande bloquée (mot : {mot})")
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "Je ne peux pas traiter cette demande. Reformulez-la, s'il vous plaît."
                }],
                "jump_to": "end"
            }
    return None

In [102]:
agent_garde = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="Tu es un assistant utile. Réponds en français, de façon courte.",
    middleware=[filtre_contenu],
)

In [103]:
for q in ["Comment faire un hack de wifi ?", "Quelle est la météo à Marrakech ?"]:
    res = agent_garde.invoke({"messages": [{"role": "user", "content": q}]})
    print("Q :", q)
    print("R :", res["messages"][-1].content, "\n")

[GARDE-FOU] demande bloquée (mot : hack)
Q : Comment faire un hack de wifi ?
R : Je ne peux pas traiter cette demande. Reformulez-la, s'il vous plaît. 

[TOOL] get_weather appelé pour Marrakech
Q : Quelle est la météo à Marrakech ?
R : Marrakech : ensoleillé, 32°C 



### Middleware 5 : Human In The Loop

In [104]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

@tool
def envoyer_email(adresse: str) -> str:
    """Envoie un e-mail à l'adresse donnée."""
    print(f"[ACTION RÉELLE] e-mail envoyé à {adresse}")
    return f"E-mail envoyé avec succès à {adresse}"

agent_hitl = create_agent(
    model=llm,
    tools=[envoyer_email],
    system_prompt="Tu es un assistant utile. Utilise l'outil pour envoyer des e-mails. Réponds en français, de façon courte.",
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"envoyer_email": True}
        ),
    ],
    checkpointer=InMemorySaver(),
)

config_hitl = {"configurable": {"thread_id": "hitl-1"}}

In [105]:
result = agent_hitl.invoke(
    {"messages": [{"role": "user", "content": "Envoie un e-mail à med@gmail.com"}]},
    config=config_hitl
)
print(result["__interrupt__"])

[Interrupt(value={'action_requests': [{'name': 'envoyer_email', 'args': {'adresse': 'med@gmail.com'}, 'description': "Tool execution requires approval\n\nTool: envoyer_email\nArgs: {'adresse': 'med@gmail.com'}"}], 'review_configs': [{'action_name': 'envoyer_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='42130707b3b57eeca43c5002c35de229', response_schema=None)]


In [106]:
result = agent_hitl.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config_hitl
)
print(result["messages"][-1].content)

[ACTION RÉELLE] e-mail envoyé à med@gmail.com
L'e-mail a été envoyé avec succès à med@gmail.com.


In [107]:
config_rejet = {"configurable": {"thread_id": "hitl-2"}}

agent_hitl.invoke(
    {"messages": [{"role": "user", "content": "Envoie un e-mail à pirate@test.com"}]},
    config=config_rejet
)
result = agent_hitl.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "Envoi refusé par l'utilisateur."}]}),
    config=config_rejet
)
print(result["messages"][-1].content)

L'e-mail a été refusé par l'utilisateur.


## Étape 5 : Chatbot RAG agentique

In [1]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatOllama(model="qwen3:4b", temperature=0, reasoning=True)

In [2]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = PyPDFDirectoryLoader("../data").load()
print("Pages chargées :", len(documents))

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print("Morceaux (chunks) :", len(chunks))
print("\nExemple :\n", chunks[0].page_content[:300])

C:\Users\bouho\AppData\Local\Temp\ipykernel_13156\3744708667.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Pages chargées : 17
Morceaux (chunks) : 24

Exemple :
 Agentic AI et MCP
Langchain, Langgraph


In [3]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="cours_agentic",
    persist_directory="../chroma_db"
)

print("Morceaux indexés :", vectorstore._collection.count())

Morceaux indexés : 24


In [5]:
resultats = vectorstore.similarity_search("Qu'est-ce que le pattern ReAct ?", k=3)

for r in resultats:
    print("- page", r.metadata.get("page"), ":", r.page_content[:200].replace("\n", " "))
    print()

- page 4 : • Insérer un humain dans la boucle, • Faire du checkpointing • LangGraph est aussi préférable pour des systèmes où la prévisibilité est  importante (Workflow déterministes) • Débogage avec LangSmith e

- page 10 : • SSE (Server Sent Event : serveur→client en streaming + POST séparés pour client→serveur • Streamable HTTP ( un seul échange HTTP full-duplex en NDJSON , HTTP/1.1 chunked ou HTTP2) AI Client https://

- page 15 : • Le service après-vente d’un distributeur d’appareils électroniques  souhaite automatiser deux types de demandes.  • La première concerne le dépannage à partir d’une question et,  éventuellement, d’u



In [6]:
@tool
def rechercher_cours(question: str) -> str:
    """Recherche dans le support de cours (Agentic AI, LangChain, LangGraph, MCP)
    et renvoie les passages pertinents avec leur numéro de page."""
    docs = vectorstore.similarity_search(question, k=4)
    return "\n\n".join(
        f"[page {d.metadata.get('page', 0) + 1}] {d.page_content}" for d in docs
    )

agent_rag = create_agent(
    model=llm,
    tools=[rechercher_cours],
    system_prompt=(
        "Tu es un assistant pédagogique. Pour toute question sur le cours "
        "(Agentic AI, LangChain, LangGraph, MCP), utilise l'outil rechercher_cours "
        "et réponds UNIQUEMENT à partir des passages trouvés, en citant les pages. "
        "Si l'information n'est pas dans le cours, dis-le clairement. "
        "Pour une simple salutation, réponds sans chercher. Réponds en français."
    ),
    checkpointer=InMemorySaver()
)

config_rag = {"configurable": {"thread_id": "rag-1"}}

In [7]:
def discuter(question):
    res = agent_rag.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config_rag
    )
    print("Q :", question)
    for m in res["messages"]:
        for tc in getattr(m, "tool_calls", []) or []:
            print("[RECHERCHE]", tc["args"])
    print("R :", res["messages"][-1].content, "\n")

In [8]:
discuter("Bonjour !")
discuter("Quels sont les modes de transport du protocole MCP ?")
discuter("Quelle est la recette de la pizza ?")

Q : Bonjour !
R : Bonjour ! Comment puis-je t'aider aujourd'hui ? 😊 

Q : Quels sont les modes de transport du protocole MCP ?
[RECHERCHE] {'question': 'Quels sont les modes de transport du protocole MCP ?'}
R : Les modes de transport du protocole MCP sont les suivants (page 11) :  
- **STDIO** (communication via l'entrée et la sortie standard)  
- **SSE** (Server Sent Events : échanges en streaming serveur→client avec des POST séparés pour client→serveur).  

Ces mécanismes permettent de standardiser la communication client-serveur dans l'écosystème d'applications IA. 

Q : Quelle est la recette de la pizza ?
[RECHERCHE] {'question': 'Quels sont les modes de transport du protocole MCP ?'}
R : La recette de la pizza n'est pas abordée dans le cours sur Agentic AI, LangChain, LangGraph ou MCP. Ce support de cours ne contient pas d'informations relatives à la cuisine ou aux recettes de pizza. 😊 



## Étape 6 : Bases de LangGraph

In [11]:
from typing import Annotated, List
from typing_extensions import TypedDict

from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

llm = ChatOllama(model="qwen3:4b", temperature=0, reasoning=True)

In [12]:
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]   # add_messages = "reducer"

@tool
def multiply(a: float, b: float) -> float:
    """Multiplie deux nombres."""
    print(f"[TOOL] multiply a={a}, b={b}")
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divise a par b."""
    print(f"[TOOL] divide a={a}, b={b}")
    return a / b

tools = [multiply, divide]
llm_with_tools = llm.bind_tools(tools)

In [13]:
def assistant(state: AgentState) -> AgentState:
    """Nœud qui appelle le LLM (avec les outils)."""
    system_message = SystemMessage(
        "Réponds à la question de l'utilisateur en utilisant les outils fournis. Réponds en français."
    )
    response = llm_with_tools.invoke([system_message] + state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState):
    """Si le LLM demande un outil -> 'continue', sinon -> 'end'."""
    dernier = state["messages"][-1]
    if not dernier.tool_calls:
        return "end"
    return "continue"

In [14]:
workflow = StateGraph(AgentState)

workflow.add_node("assistant", assistant)
workflow.add_node("tools", ToolNode(tools))

workflow.add_edge(START, "assistant")
workflow.add_conditional_edges(
    "assistant", should_continue,
    {"continue": "tools", "end": END}
)
workflow.add_edge("tools", "assistant")

graph = workflow.compile()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	assistant(assistant)
	tools(tools<hr/><small><em>__tools = multiply, divide</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> assistant;
	assistant -. &nbsp;end&nbsp; .-> __end__;
	assistant -. &nbsp;continue&nbsp; .-> tools;
	tools --> assistant;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [15]:
resp = graph.invoke({
    "messages": [HumanMessage(
        "Calcule : 6 multiplié par 7, puis divisé par 2. Puis donne-moi un proverbe."
    )]
})
print(resp["messages"][-1].content)

[TOOL] multiply a=6.0, b=7.0
[TOOL] divide a=42.0, b=2.0
Le résultat de l'opération est 21. Voici un proverbe : "L'argent ne fait pas le bonheur."


In [16]:
from langgraph.checkpoint.memory import InMemorySaver

graph_mem = workflow.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "1"}}

r = graph_mem.invoke({"messages": [HumanMessage("Mon nom est Houssam")]}, config=config)
print(r["messages"][-1].content)

Bonjour Houssam ! Comment puis-je t'aider aujourd'hui ?


In [17]:
r = graph_mem.invoke({"messages": [HumanMessage("C'est quoi mon nom ?")]}, config=config)
print(r["messages"][-1].content)

Bonjour Houssam ! Ton nom est Houssam. 😊 Comment puis-je t'aider aujourd'hui ?


In [18]:
etat = graph_mem.get_state(config).values["messages"]
for m in etat:
    print(type(m).__name__, ":", str(m.content)[:80])

HumanMessage : Mon nom est Houssam
AIMessage : Bonjour Houssam ! Comment puis-je t'aider aujourd'hui ?
HumanMessage : C'est quoi mon nom ?
AIMessage : Bonjour Houssam ! Ton nom est Houssam. 😊 Comment puis-je t'aider aujourd'hui ?


## Étape 7 : Chatbot Agentic RAG avec LangGraph

In [19]:
from typing import Annotated, List
from typing_extensions import TypedDict

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain.tools import tool
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver

llm_rag = ChatOllama(model="qwen3:4b", temperature=0, reasoning=True)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma(
    collection_name="cours_agentic",
    embedding_function=embeddings,
    persist_directory="../chroma_db"
)
print("Morceaux dans la base :", vectorstore._collection.count())

Morceaux dans la base : 24


In [20]:
@tool
def rechercher_cours(question: str) -> str:
    """Recherche dans le support de cours (Agentic AI, LangChain, LangGraph, MCP)
    et renvoie les passages pertinents avec leur numéro de page."""
    docs = vectorstore.similarity_search(question, k=6)
    return "\n\n".join(
        f"[page {d.metadata.get('page', 0) + 1}] {d.page_content}" for d in docs
    )

tools_rag = [rechercher_cours]
llm_rag_tools = llm_rag.bind_tools(tools_rag)

In [21]:
SYSTEM_RAG = (
    "Tu es un assistant pédagogique. Pour toute question sur le cours "
    "(Agentic AI, LangChain, LangGraph, MCP), utilise l'outil rechercher_cours "
    "et réponds UNIQUEMENT à partir des passages trouvés, en citant les pages. "
    "Si l'information n'est pas dans le cours, dis-le clairement. "
    "Pour une simple salutation, réponds sans chercher. Réponds en français."
)

class EtatRAG(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

def noeud_assistant(state: EtatRAG) -> EtatRAG:
    response = llm_rag_tools.invoke([SystemMessage(SYSTEM_RAG)] + state["messages"])
    return {"messages": [response]}

def doit_continuer(state: EtatRAG):
    return "continue" if state["messages"][-1].tool_calls else "end"

workflow_rag = StateGraph(EtatRAG)
workflow_rag.add_node("assistant", noeud_assistant)
workflow_rag.add_node("tools", ToolNode(tools_rag))
workflow_rag.add_edge(START, "assistant")
workflow_rag.add_conditional_edges("assistant", doit_continuer, {"continue": "tools", "end": END})
workflow_rag.add_edge("tools", "assistant")

graph_rag = workflow_rag.compile(checkpointer=InMemorySaver())
print(graph_rag.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	assistant(assistant)
	tools(tools<hr/><small><em>__tools = rechercher_cours</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> assistant;
	assistant -. &nbsp;end&nbsp; .-> __end__;
	assistant -. &nbsp;continue&nbsp; .-> tools;
	tools --> assistant;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [24]:
def discuter_rag(question, thread_id="rag-1"):
    config = {"configurable": {"thread_id": thread_id}}
    print("Q :", question)
    for event in graph_rag.stream({"messages": [HumanMessage(question)]},
                                  config=config, stream_mode="updates"):
        for noeud, maj in event.items():
            msg = maj["messages"][-1]
            if noeud == "assistant" and msg.tool_calls:
                tc = msg.tool_calls[0]
                print(f"  [assistant] -> outil {tc['name']} {tc['args']}")
            elif noeud == "tools":
                print(f"  [tools] {len(msg.content)} caractères renvoyés")
            else:
                print(f"  [assistant] réponse : {msg.content}")
    print()

In [25]:
discuter_rag("Bonjour !")
discuter_rag("Quels sont les modes de transport du protocole MCP ?")
discuter_rag("Quelle est la recette de la pizza ?")

Q : Bonjour !
  [assistant] réponse : Bonjour ! Comment puis-je t'aider aujourd'hui ? 😊

Q : Quels sont les modes de transport du protocole MCP ?
  [assistant] -> outil rechercher_cours {'question': 'Quels sont les modes de transport du protocole MCP ?'}
  [tools] 2472 caractères renvoyés
  [assistant] réponse : Les modes de transport du protocole MCP sont **STDIO** (communication via l'entrée et la sortie standard) et **SSE** (Server Sent Event : serveur→client en streaming + POST séparés pour client→serveur). Ces informations sont spécifiées sur la page 11.

Q : Quelle est la recette de la pizza ?
  [assistant] réponse : La question "Quelle est la recette de la pizza ?" n'est pas abordée dans le cours concernant l'Agentic AI, LangChain, LangGraph ou MCP. Les supports de cours fournis ne contiennent aucune information sur les recettes de pizza. 

Je peux vous aider à trouver d'autres informations sur le sujet si vous le souhaitez, mais elles ne sont pas incluses dans ce cours. 😊

